In [28]:
import os
import certifi
import requests
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain.agents import create_react_agent, AgentExecutor

In [42]:
# =========================================
# LOAD ENVIRONMENT VARIABLES
# =========================================

os.environ['SSL_CERT_FILE'] = certifi.where()
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [30]:
search_tool = TavilySearchResults(max_results=2)
result = search_tool.invoke("Give me the latest news about AI")
result

[{'url': 'https://en.wikipedia.org/wiki/Artificial_intelligence',
  'content': 'Edwards, Benj (17 May 2023). "Poll: AI poses risk to humanity, according to majority of Americans". _Ars Technica_. Archived from the original on 19 June 2023. Retrieved 19 June 2023.\n   Fearn, Nicholas (2007). _The Latest Answers to the Oldest Questions: A Philosophical Adventure with the World\'s Greatest Thinkers_. New York: Grove Press. ISBN "ISBN (identifier)")978-0-8021-1839-4.\n   Ford, Martin; Colvin, Geoff (6 September 2015). "Will robots create more jobs than they destroy?". _The Guardian_. Archived from the original on 16 June 2018. Retrieved 13 January 2018.\n   Fox News (2023). "Fox News Poll"(PDF). Fox News. Archived(PDF) from the original on 12 May 2023. Retrieved 19 June 2023. [...] 351.   ↑"Council of Europe opens first ever global treaty on AI for signature". _Council of Europe_. 5 September 2024. Archived from the original on 17 September 2024. Retrieved 17 September 2024.\n352.   ↑Edwar

In [49]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """
    # Removed the comma and joined them into a single string
    url = f"https://api.weatherstack.com/current?access_key={WEATHERSTACK_API_KEY}&query={city}"

    response = requests.get(url)
    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}."

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [50]:
# LLM

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0, openai_api_key=OPENAI_API_KEY)

In [51]:
response = llm.invoke("tell me a joke about AI")
response

AIMessage(content='Why did the AI go broke?\n\nBecause it kept losing its cache!', response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 13, 'total_tokens': 27, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini', 'system_fingerprint': 'fp_f3e96e7d97', 'finish_reason': 'stop', 'logprobs': None}, id='run-d38a7e7d-197b-45d5-b1f2-c09c79f9836e-0')

In [52]:
# PROMPT

prompt = hub.pull("hwchase17/react")

c:\Users\knagr\Downloads\Langchain-Agent\langagent\Lib\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [53]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [54]:
#TOOL


tools = [search_tool, get_weather_data]

In [55]:
#CREATE AGENT

agent = create_react_agent(llm=llm, tools=tools, prompt=prompt)


In [56]:
#EXECUTOR

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [57]:
#RUN

response = agent_executor.invoke({
    "input":(
    "Find the capital of india"
    "and then find its current weather."
    )
})





> Entering new AgentExecutor chain...
I need to find the capital of India first. The capital is a well-known fact, so I can answer that directly. After that, I will check the current weather in that city. 

Action: None needed for the capital, as I know it is New Delhi.
Action Input: NoneNone needed for the capital, as I know it is New Delhi. is not a valid tool, try one of [tavily_search_results_json, get_weather_data].I need to find the capital of India first. The capital is a well-known fact, so I can answer that directly. After that, I will check the current weather in that city.

Action: None needed for the capital, as I know it is New Delhi.
Action Input: NoneNone needed for the capital, as I know it is New Delhi. is not a valid tool, try one of [tavily_search_results_json, get_weather_data].I need to find the capital of India first. The capital is a well-known fact, so I can answer that directly. After that, I will check the current weather in that city.

Action: None needed f

In [58]:
print(response['output'])

The capital of India is New Delhi, and its current weather is sunny with a temperature of 28°C and humidity at 40%.
